# ENSU
ENSU (INEGI) - Percepción de seguridad por alcaldía de la CDMX.
Lee, uno por uno, todos los ZIP de "Tabulados básicos" que ya se tienen guardados en Google Drive (carpeta "seguridad"), y de cada ZIP extrae
el archivo de Excel cuyo nombre empieza con "1." (ej. "1.Bas_percepcion_seguridad_mar_2025_est.xlsx"), abre la pestaña "1.7"
 y registra los valores de las 16 alcaldías, respetando la estructura
 Región -> Alcaldía -> Seguro/Inseguro que tiene el archivo original.

 Al final obtienes un único DataFrame (df_final) con una fila por
 alcaldía-trimestre.

In [6]:
#Librerías y montaje de Google Drive
import os
import re
import glob
import zipfile
import shutil
import pandas as pd
from openpyxl import load_workbook

from google.colab import drive
drive.mount('/content/drive')

# Carpeta donde están tus ZIP descargados de INEGI.
# Se busca de forma insensible a mayúsculas/minúsculas porque Linux (Colab)
# sí distingue "seguridad" de "Seguridad", a diferencia de Windows.
def encontrar_carpeta(nombre_carpeta="seguridad", base="/content/drive/MyDrive"):
    ruta_directa = os.path.join(base, nombre_carpeta)
    if os.path.isdir(ruta_directa):
        return ruta_directa
    for entrada in os.listdir(base):
        ruta = os.path.join(base, entrada)
        if entrada.lower() == nombre_carpeta.lower() and os.path.isdir(ruta):
            print(f"[Info] Se usó la carpeta '{entrada}' (coincide con '{nombre_carpeta}' ignorando mayúsculas).")
            return ruta
    raise FileNotFoundError(
        f"No se encontró ninguna carpeta llamada '{nombre_carpeta}' (ni variantes de "
        f"mayúsculas/minúsculas) dentro de {base}. Carpetas disponibles: "
        f"{[e for e in os.listdir(base) if os.path.isdir(os.path.join(base, e))]}"
    )

FOLDER_PATH = encontrar_carpeta("seguridad")

# Carpeta de trabajo temporal (dentro de la sesión de Colab, no en Drive)
TEMP_DIR = "/content/ensu_tmp"
os.makedirs(TEMP_DIR, exist_ok=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[Info] Se usó la carpeta 'Seguridad' (coincide con 'seguridad' ignorando mayúsculas).


In [7]:
# -----------------------------------------------------------------------
# PASO 1. Utilidades para identificar el periodo (año-trimestre) a partir
#          del nombre del archivo ZIP
# -----------------------------------------------------------------------

MESES_A_TRIMESTRE = {
    "enero": "T1", "febrero": "T1", "marzo": "T1",
    "abril": "T2", "mayo": "T2", "junio": "T2",
    "julio": "T3", "agosto": "T3", "septiembre": "T3", "setiembre": "T3",
    "octubre": "T4", "noviembre": "T4", "diciembre": "T4",
    # abreviaturas (las que realmente usa INEGI en el nombre del excel interno,
    # p. ej. '1.Bas_percepcion_seguridad_mar_2025_est.xlsx')
    "ene": "T1", "feb": "T1", "mar": "T1",
    "abr": "T2", "may": "T2", "jun": "T2",
    "jul": "T3", "ago": "T3", "sep": "T3", "sept": "T3",
    "oct": "T4", "nov": "T4", "dic": "T4",
}

# Nombre "completo" del mes para dejarlo bonito en el DataFrame, aunque el
# archivo haya traído la abreviatura
MES_COMPLETO = {
    "ene": "enero", "feb": "febrero", "mar": "marzo", "abr": "abril",
    "may": "mayo", "jun": "junio", "jul": "julio", "ago": "agosto",
    "sep": "septiembre", "sept": "septiembre", "oct": "octubre",
    "nov": "noviembre", "dic": "diciembre",
}

def extraer_periodo(nombre_archivo):
    """
    A partir de un nombre de archivo (puede ser el del .zip o el del .xlsx
    interno, p. ej. '1.Bas_percepcion_seguridad_mar_2025_est.xlsx') intenta
    identificar año y mes/trimestre. Se probaron ambos nombres porque en
    algunos casos el .zip tiene un nombre distinto al del excel que contiene.
    Devuelve (anio, mes_completo, trimestre_clave) o (None, None, None) si
    no logra identificar nada.
    """
    nombre_low = nombre_archivo.lower()

    mes_encontrado = None
    # Ordenamos por longitud descendente para que "septiembre" no sea
    # "atrapado" primero como "sep" y viceversa
    for mes in sorted(MESES_A_TRIMESTRE, key=len, reverse=True):
        if re.search(rf"(?<![a-záéíóúñ]){mes}(?![a-záéíóúñ])", nombre_low):
            mes_encontrado = mes
            break

    anio_match = re.search(r"(20\d{2})", nombre_low)
    anio_encontrado = int(anio_match.group(1)) if anio_match else None

    if mes_encontrado is None or anio_encontrado is None:
        return None, None, None

    mes_completo = MES_COMPLETO.get(mes_encontrado, mes_encontrado)
    return anio_encontrado, mes_completo, MESES_A_TRIMESTRE[mes_encontrado]



In [8]:
# PASO 2. Dentro de un ZIP, localizar el excel que empieza con "1."
# -----------------------------------------------------------------------

def obtener_excel_uno(zip_path, destino):
    """
    Abre el zip y busca el excel de la SECCIÓN 1 (Percepción de seguridad
    pública), sin importar si INEGI lo nombró con numeración arábiga
    ('1.Bas_percepcion_seguridad_mar_2025_est.xlsx', trimestres recientes)
    o con numeración romana ('I_percepcion_dic_2020_est.xlsx', trimestres
    de 2020-2021). El patrón exige que justo después del '1' o la 'I' venga
    un '.' o un '_', para no confundirla con 'II_...', 'III_...', '10.Bas_...'
    o '14.Bas_...'.
    Devuelve la ruta local del xlsx extraído, o None si no lo encuentra.
    """
    patron_seccion1 = re.compile(r'^[1I][._]', re.IGNORECASE)

    with zipfile.ZipFile(zip_path) as z:
        nombres_xlsx = [n for n in z.namelist() if n.lower().endswith(".xlsx")]

        candidatos = [
            n for n in nombres_xlsx
            if patron_seccion1.match(os.path.basename(n).strip())
        ]

        if not candidatos:
            print(f"  [Aviso] No se encontró el excel de la sección 1 en {os.path.basename(zip_path)}.")
            print(f"          Archivos .xlsx dentro del zip: {[os.path.basename(n) for n in nombres_xlsx]}")
            return None

        # Si hubiera más de uno, tomamos el primero en orden alfabético
        candidatos.sort()
        elegido = candidatos[0]

        z.extract(elegido, destino)
        return os.path.join(destino, elegido)

In [9]:
# PASO 3. Dentro del excel, localizar la pestaña "1.7"
# -----------------------------------------------------------------------

def obtener_hoja_1_7(xlsx_path):
    """
    Devuelve el objeto worksheet correspondiente a la pestaña "1.7".
    Si el nombre exacto de la pestaña no existiera (por si INEGI cambia
    la numeración en algún trimestre), se usa como respaldo el
    contenido característico del cuadro: la celda "Ciudad de México"
    seguida más abajo de "Gustavo A. Madero".
    """
    wb = load_workbook(xlsx_path, data_only=True, read_only=True)

    # Intento 1: nombre exacto de la pestaña
    for nombre_hoja in wb.sheetnames:
        if nombre_hoja.strip() == "1.7":
            return wb[nombre_hoja]

    # Intento 2 (respaldo): buscar la huella del cuadro en cualquier hoja
    for nombre_hoja in wb.sheetnames:
        ws = wb[nombre_hoja]
        texto_muestra = []
        for fila in ws.iter_rows(min_row=1, max_row=min(ws.max_row, 100), values_only=True):
            texto_muestra.extend(str(c) for c in fila if c is not None)
        texto_muestra = " ".join(texto_muestra)
        if "Ciudad de México" in texto_muestra and "Gustavo A. Madero" in texto_muestra:
            return ws

    return None


In [10]:
# PASO 4. Parsear el cuadro de alcaldías respetando su estructura original
#          Región  -> Alcaldía (con población) -> Seguro / Inseguro
# -----------------------------------------------------------------------

# Lista de control: las 16 alcaldías tal como aparecen en el cuadro de INEGI
ALCALDIAS_CDMX = [
    "Gustavo A. Madero", "Iztacalco", "Venustiano Carranza", "Cuauhtémoc",
    "Benito Juárez", "Coyoacán", "La Magdalena Contreras", "Tlalpan",
    "Iztapalapa", "Milpa Alta", "Tláhuac", "Xochimilco",
    "Álvaro Obregón", "Miguel Hidalgo", "Azcapotzalco", "Cuajimalpa de Morelos",
]

def parsear_hoja_alcaldias(ws, anio, mes, trimestre_clave):
    """
    Recorre la hoja fila por fila a partir del renglón "Ciudad de México"
    y arma una lista de diccionarios (uno por alcaldía) con:
      anio, mes, trimestre, region, alcaldia, poblacion_18mas,
      seguro_abs, seguro_pct, inseguro_abs, inseguro_pct

    La lógica reproduce exactamente el layout del archivo de INEGI:
      Fila "Región XXX"        -> solo texto en columna A (marca la región)
      Fila "<Alcaldía>"        -> texto en A + población total en columna B
      Fila "Seguro"            -> valor absoluto en D, porcentaje en E
      Fila "Inseguro"          -> valor absoluto en D, porcentaje en E
    """
    filas = list(ws.iter_rows(values_only=True))

    # 4.1 localizar la fila ancla "Ciudad de México"
    idx_inicio = None
    for i, fila in enumerate(filas):
        if fila and fila[0] and str(fila[0]).strip() == "Ciudad de México":
            idx_inicio = i
            break

    if idx_inicio is None:
        print("  [Aviso] No se encontró el renglón 'Ciudad de México' en la hoja 1.7")
        return []

    resultados = []
    region_actual = None
    alcaldia_actual = None
    poblacion_actual = None

    for fila in filas[idx_inicio + 1:]:
        col_a = fila[0] if len(fila) > 0 else None
        col_b = fila[1] if len(fila) > 1 else None
        col_d = fila[3] if len(fila) > 3 else None
        col_e = fila[4] if len(fila) > 4 else None

        if col_a is None or str(col_a).strip() == "":
            continue

        texto = str(col_a).strip()

        # 4.2 fila de región
        if texto.lower().startswith("región"):
            region_actual = texto
            continue

        # 4.3 fila de dato Seguro / Inseguro
        if texto in ("Seguro", "Inseguro") and alcaldia_actual is not None:
            campo = "seguro" if texto == "Seguro" else "inseguro"
            registro = next((r for r in resultados if r["alcaldia"] == alcaldia_actual), None)
            if registro is None:
                registro = {
                    "anio": anio,
                    "mes": mes,
                    "trimestre": trimestre_clave,
                    "region": region_actual,
                    "alcaldia": alcaldia_actual,
                    "poblacion_18mas": poblacion_actual,
                }
                resultados.append(registro)
            registro[f"{campo}_abs"] = col_d
            registro[f"{campo}_pct"] = col_e

            # Nos detenemos SOLO cuando las 16 alcaldías ya tienen sus
            # dos valores (seguro e inseguro) completos; así no se corta
            # a media fila la última alcaldía del cuadro.
            completas = [
                r for r in resultados
                if "seguro_pct" in r and "inseguro_pct" in r
            ]
            if len(completas) == len(ALCALDIAS_CDMX):
                break
            continue

        # 4.4 fila de encabezado de alcaldía
        if texto in ALCALDIAS_CDMX:
            alcaldia_actual = texto
            poblacion_actual = col_b

    return resultados



In [11]:
# -----------------------------------------------------------------------
# PASO 5. Orquestar todo: recorrer cada ZIP de la carpeta de Drive
# -----------------------------------------------------------------------

def procesar_carpeta(folder_path=FOLDER_PATH):
    zips = sorted(glob.glob(os.path.join(folder_path, "*.zip")))
    print(f"Se encontraron {len(zips)} archivos .zip en {folder_path}\n")

    todos_los_registros = []

    for zip_path in zips:
        nombre_zip = os.path.basename(zip_path)
        print(f"Procesando: {nombre_zip}")

        # Carpeta temporal única para este zip (evita mezclar archivos
        # de un trimestre con los de otro)
        destino = os.path.join(TEMP_DIR, os.path.splitext(nombre_zip)[0])
        os.makedirs(destino, exist_ok=True)

        xlsx_path = obtener_excel_uno(zip_path, destino)
        if xlsx_path is None:
            continue

        # 1) intentamos identificar el periodo con el nombre del zip
        anio, mes, trimestre_clave = extraer_periodo(nombre_zip)
        # 2) si no se pudo, lo intentamos con el nombre del excel interno,
        #    que sí sigue el patrón confirmado "..._mar_2025_est.xlsx"
        if anio is None:
            anio, mes, trimestre_clave = extraer_periodo(os.path.basename(xlsx_path))

        if anio is None:
            print(f"  [Aviso] No se pudo identificar año/mes ni en '{nombre_zip}' "
                  f"ni en '{os.path.basename(xlsx_path)}'; se omite este archivo.")
            continue

        hoja = obtener_hoja_1_7(xlsx_path)
        if hoja is None:
            print(f"  [Aviso] No se encontró la pestaña 1.7 en {os.path.basename(xlsx_path)}")
            continue

        registros = parsear_hoja_alcaldias(hoja, anio, mes, trimestre_clave)
        if not registros:
            print(f"  [Aviso] La pestaña 1.7 se abrió pero no se pudo extraer "
                  f"ninguna alcaldía (revisa si el layout cambió en este trimestre).")
        else:
            print(f"  OK: {len(registros)} alcaldías extraídas ({mes} {anio})")
        print()
        todos_los_registros.extend(registros)

    return todos_los_registros



In [12]:
# PASO 6. Ejecutar y construir el DataFrame final
# -----------------------------------------------------------------------

registros = procesar_carpeta(FOLDER_PATH)

if not registros:
    raise SystemExit(
        "\nNo se extrajo ningún registro. Revisa los mensajes '[Aviso]' de arriba:\n"
        "  - Si dice 'No se pudo identificar año/mes': el nombre del zip/excel no "
        "trae un mes en español ni un año de 4 dígitos reconocible.\n"
        "  - Si dice 'No se encontró la pestaña 1.7': el archivo no tiene esa "
        "pestaña o el nombre de pestaña cambió ese trimestre.\n"
        "  - Si dice 'no se pudo extraer ninguna alcaldía': el layout de esa "
        "hoja específica es distinto al esperado.\n"
        "Copia el bloque de mensajes completo si quieres que lo revisemos."
    )

df_final = pd.DataFrame(registros)

# Orden de columnas legible
columnas_orden = [
    "anio", "mes", "trimestre", "region", "alcaldia", "poblacion_18mas",
    "seguro_abs", "seguro_pct", "inseguro_abs", "inseguro_pct",
]
df_final = df_final[[c for c in columnas_orden if c in df_final.columns]]

# Orden lógico de filas: por fecha y luego en el mismo orden en que
# aparecen las alcaldías en el documento original de INEGI
df_final["alcaldia"] = pd.Categorical(df_final["alcaldia"], categories=ALCALDIAS_CDMX, ordered=True)
df_final = df_final.sort_values(["anio", "trimestre", "alcaldia"]).reset_index(drop=True)

print(df_final.head(20))
print(f"\nTotal de filas en df_final: {len(df_final)}")
###Renombrar
df_final["trimestre"] = df_final["trimestre"].replace({
    "T1": 1,
    "T2": 2,
    "T3": 3,
    "T4": 4
})

Se encontraron 25 archivos .zip en /content/drive/MyDrive/Seguridad

Procesando: ensu_tabulados_basicos_diciembre_2020_estimacion_xlsx.zip
  OK: 16 alcaldías extraídas (diciembre 2020)

Procesando: ensu_tabulados_basicos_diciembre_2021_estimacion_xlsx.zip
  OK: 16 alcaldías extraídas (diciembre 2021)

Procesando: ensu_tabulados_basicos_diciembre_2022_estimacion_xlsx.zip
  OK: 16 alcaldías extraídas (diciembre 2022)

Procesando: ensu_tabulados_basicos_diciembre_2023_estimacion_xlsx.zip
  OK: 16 alcaldías extraídas (diciembre 2023)

Procesando: ensu_tabulados_basicos_diciembre_2024_estimacion_xlsx.zip
  OK: 16 alcaldías extraídas (diciembre 2024)

Procesando: ensu_tabulados_basicos_diciembre_2025_estimacion_xlsx.zip
  OK: 16 alcaldías extraídas (diciembre 2025)

Procesando: ensu_tabulados_basicos_junio_2021_estimacion_xlsx.zip
  OK: 16 alcaldías extraídas (junio 2021)

Procesando: ensu_tabulados_basicos_junio_2022_estimacion_xlsx.zip
  OK: 16 alcaldías extraídas (junio 2022)

Procesando:

/tmp/ipykernel_1196/1860076385.py:35: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_final["trimestre"] = df_final["trimestre"].replace({


In [13]:
# -----------------------------------------------------------------------
# PASO 7 (opcional). Guardar el resultado como CSV en Drive
# -----------------------------------------------------------------------
ruta_salida = os.path.join(FOLDER_PATH, "percepcion_seguridad_alcaldias_cdmx.csv")
df_final.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
print(f"\nCSV guardado en: {ruta_salida}")


CSV guardado en: /content/drive/MyDrive/Seguridad/percepcion_seguridad_alcaldias_cdmx.csv
